In [ ]:
!pip install -q kagglehub

: 

In [ ]:
import kagglehub

path = kagglehub.dataset_download("lavuluriliketh/oral-cancer-dataset")

print(path)

: 

In [ ]:
import os

for root, dirs, files in os.walk("/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1"):
    print(root, len(files))

/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1 0
/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1/data 0
/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1/data/non-cancer 242
/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1/data/cancer 287


In [ ]:
import os
import numpy as np
from sklearn.model_selection import train_test_split

data_path = "/root/.cache/kagglehub/datasets/lavuluriliketh/oral-cancer-dataset/versions/1/data"

images = []
labels = []

for label, class_name in enumerate(["non-cancer", "cancer"]):
    folder = os.path.join(data_path, class_name)

    for file in os.listdir(folder):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            images.append(os.path.join(folder, file))
            labels.append(label)

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

Training images: 423
Testing images: 106


In [ ]:
from skimage.io import imread
from skimage.transform import resize
from skimage.feature import hog
import numpy as np

def extract_features(image_paths):
    features = []
    for path in image_paths:
        image = imread(path, as_gray=True)
        image = resize(image, (128, 128))

        feature = hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2)
        )

        features.append(feature)

    return np.array(features)

X_train_features = extract_features(X_train)
X_test_features = extract_features(X_test)

print("Training shape:", X_train_features.shape)
print("Testing shape:", X_test_features.shape)

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Training shape: (423, 8100)
Testing shape: (106, 8100)


In [ ]:
from sklearn.svm import SVC

model = SVC(kernel='rbf')

model.fit(X_train_features, y_train)

y_pred = model.predict(X_test_features)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC-AUC
y_prob = model.decision_function(X_test_features)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy    :", round(accuracy, 3))
print("Precision   :", round(precision, 3))
print("Recall      :", round(recall, 3))
print("F1-score    :", round(f1, 3))
print("ROC-AUC     :", round(roc_auc, 3))

print("\nConfusion Matrix:")
print(cm)

Accuracy    : 0.736
Precision   : 0.692
Recall      : 0.931
F1-score    : 0.794
ROC-AUC     : 0.792

Confusion Matrix:
[[24 24]
 [ 4 54]]


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

def show_metrics():

    # Get TN, FP, FN, TP
    tn, fp, fn, tp = cm.ravel()

    # Create Confusion Matrix image
    matrix = np.array([
        [tn, fp],
        [fn, tp]
    ])

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.imshow(matrix)

    labels = [
        [f"TN\n{tn}", f"FP\n{fp}"],
        [f"FN\n{fn}", f"TP\n{tp}"]
    ]

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i,
                labels[i][j],
                ha="center",
                va="center",
                fontsize=14
            )

    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Non-Cancer", "Cancer"])
    ax.set_yticklabels(["Non-Cancer", "Cancer"])
    ax.set_title("Confusion Matrix")

    plt.tight_layout()

    return (
        f"{accuracy:.3f}",
        f"{precision:.3f}",
        f"{recall:.3f}",
        f"{f1:.3f}",
        f"{roc_auc:.3f}",
        fig
    )


with gr.Blocks() as demo:

    gr.Markdown("# 🩺 Oral Cancer Classification")
    gr.Markdown("### Model Performance Metrics")

    button = gr.Button("Show Results")

    accuracy_output = gr.Textbox(label="Accuracy")
    precision_output = gr.Textbox(label="Precision")
    recall_output = gr.Textbox(label="Recall")
    f1_output = gr.Textbox(label="F1-score")
    roc_output = gr.Textbox(label="ROC-AUC")

    confusion_output = gr.Plot(label="Confusion Matrix")

    button.click(
        fn=show_metrics,
        outputs=[
            accuracy_output,
            precision_output,
            recall_output,
            f1_output,
            roc_output,
            confusion_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4293ce7fcfccdbbd7c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
